In [1]:
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from keras.models import Sequential, load_model
from keras.layers import Embedding, Dense, LSTM, Dropout, TimeDistributed, RepeatVector

2024-10-06 15:10:38.742740: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [31]:
df = pd.read_csv('../dataset/fra-eng/eng-french.csv')
df.head()

,English words/sentences,French words/sentences
0,Hi.,Salut!
1,Run!,Cours !
2,Run!,Courez !
3,Who?,Qui ?
4,Wow!,Ça alors !


In [32]:
fra = df['French words/sentences']

In [34]:
from collections import Counter

fra_word_counter = Counter([word for sentence in fra for word in sentence.split()])
print("Total count of French words:",len([word for sentence in fra for word in sentence.split()]))
print("Count of distinct French words:",len(fra_word_counter))
print("10 most common French words:",list(zip(*fra_word_counter.most_common(10)))[0])

Total count of French words: 1177832
Count of distinct French words: 44918
10 most common French words: ('de', 'Je', '?', 'pas', 'que', 'à', 'ne', 'la', 'le', 'Il')


In [3]:
def word_count(line):
  return len(line.split())

In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer

def create_tokenizer(sentences):
  tokenizer = Tokenizer()
  tokenizer.fit_on_texts(sentences)
  return tokenizer

In [5]:
def max_sentence_length(lines):
  return max(len(sentence.split()) for sentence in lines)

In [6]:
def encode_sequences(tokenizer,sentences,max_sent_len):
  text_to_seq = tokenizer.texts_to_sequences(sentences) # encode sequences with integers
  text_pad_seq = pad_sequences(text_to_seq,maxlen=max_sent_len,padding='post') # pad sequences with 0
  return text_pad_seq

English vocabulary size: 2
Maximum length of English sentences: 1


In [35]:
# Prepare French tokenizer
fra_tokenizer = create_tokenizer(fra)
fra_vocab_size = len(fra_tokenizer.word_index) + 1
max_fra_sent_len = max_sentence_length(fra)
print("French vocabulary size:", fra_vocab_size)
print("Maximum length of French sentences:", max_fra_sent_len)

French vocabulary size: 30661
Maximum length of French sentences: 55


In [17]:

from keras.models import Sequential, load_model
translator_model = load_model('english_to_french_translator.h5')

In [18]:
translator_model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 22, 256)        │     3,720,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ (None, 256)            │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat_vector_3 (RepeatVector)  │ (None, 22, 256)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 22, 256)        │       525,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_6              │ (None, 22, 1024)       │       263,168 │
│ (TimeDistributed)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 22, 1024)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_7              │ (None, 22, 30661)      │    31,427,525 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 36,461,511 (139.09 MB)

 Trainable params: 36,461,509 (139.09 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [58]:
# Perform encoding sequences
# Prepare English tokenizer
inputText = "Hi."
df_input = pd.DataFrame([inputText])
eng = df_input[0]
eng_tokenizer = create_tokenizer(eng)
eng_vocab_size = len(eng_tokenizer.word_index) + 1
max_eng_sent_len = max_sentence_length(eng)
print("English vocabulary size:", eng_vocab_size)
print("Maximum length of English sentences:", max_eng_sent_len)
txtEncode = encode_sequences(eng_tokenizer, eng, max_eng_sent_len)
txtEncode

English vocabulary size: 2
Maximum length of English sentences: 1


array([[1]], dtype=int32)

In [59]:
test_predictions = translator_model.predict(txtEncode)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 132ms/step


In [60]:
test_predictions

array([[[1.1803437e-03, 8.8733751e-03, 1.2329804e-03, ...,
         1.5069169e-05, 1.5819811e-05, 1.5920079e-05],
        [2.4670154e-02, 9.9598803e-03, 4.2520193e-03, ...,
         4.8453321e-06, 5.3203285e-06, 5.4057505e-06],
        [4.7409782e-01, 1.9541723e-03, 5.5775894e-03, ...,
         1.9452639e-06, 2.0999043e-06, 2.3081905e-06],
        ...,
        [9.9996555e-01, 2.1597077e-07, 1.9492579e-06, ...,
         2.1114180e-12, 2.1035153e-12, 2.8568300e-12],
        [9.9996603e-01, 2.1436969e-07, 1.9324375e-06, ...,
         2.0825737e-12, 2.0748223e-12, 2.8183721e-12],
        [9.9996626e-01, 2.1297390e-07, 1.9179774e-06, ...,
         2.0582485e-12, 2.0506192e-12, 2.7859100e-12]]], dtype=float32)

In [52]:
def convert_pred_to_sent(input_seq):
    sent = ''
    for idx in input_seq:
      if idx:
        sent += fra_tokenizer.index_word[idx] + ' '
    sent = sent[:-1]
    return sent

In [46]:
def convert_idx_to_sent(input_seq,tokenizer):
    sent = ''
    for idx in input_seq:
      if idx:
        sent += tokenizer.index_word[idx] + ' '
    return sent

In [61]:
orig_eng_text = []
orig_fra_text = []
pred_fra_text = []

for i in range(len(txtEncode)):
  orig_eng_text.append(convert_idx_to_sent(txtEncode[i],eng_tokenizer))
  pred_fra_text.append(convert_pred_to_sent(np.argmax(test_predictions[i],axis=1)))
  # orig_fra_text.append(convert_idx_to_sent(orig_fra_sent[i],fra_tokenizer))


In [62]:
predictions = pd.DataFrame()
predictions['Original English Sentence'] = orig_eng_text
# predictions['Original French Sentence'] = orig_fra_text
predictions['Predicted French Sentence'] = pred_fra_text  
predictions

,Original English Sentence,Predicted French Sentence
0,hi,vous vous
